In [ ]:
# 1. Install Unsloth and core dependencies
!pip install unsloth
!pip install --no-deps "xformers<0.0.28" "trl<0.10.0" peft accelerate bitsandbytes

# 2. Install Vision & Data dependencies
!pip install qwen-vl-utils datasets wandb

# 3. CRITICAL: Force install the latest transformers from GitHub to prevent Qwen2_VL mapping errors
!pip uninstall -y transformers
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
!pip uninstall -y trl
!pip install git+https://github.com/huggingface/trl.git

In [ ]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

In [ ]:
# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

import re
import json

def extract_json(text):
    """Safely extracts JSON from model completions that might include markdown."""
    text = text.strip()
    json_match = re.search(r'```json\n(.*?)\n```', text, re.DOTALL)
    if json_match:
        text = json_match.group(1)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def robust_float(val):
    """
    Safely casts strings, integers, and scientific notation (3e+12) to floats.
    Strips LLM hallucinations like stray spaces or commas.
    """
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        try:
            clean_val = val.strip().replace(',', '')
            return float(clean_val)
        except ValueError:
            raise ValueError(f"Cannot parse {val} to float")
    raise ValueError("Not a valid type for float conversion")


def clamp(value, low=0.0, high=1.0):
    """Clamp a value to [low, high]. Used to keep reward components bounded."""
    return max(low, min(high, value))

In [ ]:
# ==========================================
# 2. REWARD FUNCTIONS
#
# ARCHITECTURE (aligned to Spec-Align paper, Table 2):
#
# Layer 0 — Integrity Checks (hard gates, no downstream reward if failed)
#   R_f : Format  — is the output valid JSON?            {0, -2}
#   R_e : Schema  — are top-level keys correct?          {0.5, -1}
#
# Layer 1 — Semantic-Spec Reward  R_sem  (continuous, summed, normalised to [0,1])
#   topology_gate_reward : chart-type / panel layout / count  (Pass/Fail gate)
#   coord_reward         : coordinate system classification   [0,1]
#   domain_reward        : axis range / categorical domain    [0,1]
#   series_label_reward  : legend / series name alignment     [0,1]
#   data_fidelity_reward : per-point numeric accuracy (Data/Func) [0,1]
#
# KEY CHANGES vs old version:
#   1. All rewards are clamped to [0, 1] — no unbounded negatives.
#      This is the primary fix for KL explosion.
#   2. topology_gate is a true binary gate: if topology fails,
#      R_sem = 0 entirely.  Mirrors the paper exactly.
#   3. Anti-looping penalty is clamped so a badly hallucinated array
#      cannot drive the reward below 0 (which caused large gradient spikes).
#   4. format_reward returns {0, -2} only — no +1 bonus that inflates variance.
#      A parseable JSON is the minimum bar; extra credit comes from R_sem/R_code.
# ==========================================


# ------------------------------------------------------------------
# LAYER 0: INTEGRITY CHECKS
# ------------------------------------------------------------------

def format_reward_func(completions, **kwargs):
    """
    R_f — Format integrity check.
    Mirrors paper: {0, -2}.  A parseable JSON scores 0 (neutral baseline);
    an unparseable completion scores -2 (hard penalty).
    This keeps the reward scale stable — format correctness is a prerequisite,
    not a bonus, so we don't inflate the mean reward with +1 here.
    """
    rewards = []
    for comp in completions:
        parsed = extract_json(comp)
        rewards.append(0.0 if parsed is not None else -2.0)
    return rewards


def schema_reward_func(completions, **kwargs):
    """
    R_e — Schema / execution check.
    Mirrors paper: {0.5, -1}.
    +0.5 if all required top-level keys are present (spec is structurally executable).
    -1.0 if the model hallucinates forbidden legacy keys.
    Clamped to [-1, 0.5] — no unbounded downside.
    """
    required_keys = {"title", "panel_count", "panel_layout", "panels", "chart_type"}
    forbidden_keys = {"math", "rel", "stats", "trend"}
    rewards = []

    for comp in completions:
        parsed = extract_json(comp)
        if not parsed:
            rewards.append(0.0)
            continue

        score = 0.5 if required_keys.issubset(parsed.keys()) else 0.0

        comp_str = str(comp).lower()
        if any(f'"{fk}"' in comp_str for fk in forbidden_keys):
            score -= 1.0

        rewards.append(max(-1.0, score))  # clamp lower bound
    return rewards


# ------------------------------------------------------------------
# LAYER 1: SEMANTIC-SPEC REWARD  (R_sem)
# Each sub-component returns a value in [0, 1].
# The topology gate is evaluated first; if it fails, R_sem = 0.
# ------------------------------------------------------------------

def _topology_passes(parsed, target):
    """
    Binary topology gate (paper: Pass / {3}).
    Checks: chart_type, panel_count, panel_layout, and per-panel topo.type.
    Returns True only if ALL structural dimensions match.
    """
    if not parsed or not target:
        return False
    if parsed.get("chart_type") != target.get("chart_type"):
        return False
    if parsed.get("panel_count") != target.get("panel_count"):
        return False
    if parsed.get("panel_layout") != target.get("panel_layout"):
        return False
    # Per-panel topology type
    p_panels = parsed.get("panels", [])
    t_panels = target.get("panels", [])
    if len(p_panels) != len(t_panels):
        return False
    for pp, tp in zip(p_panels, t_panels):
        if pp.get("topo", {}).get("type") != tp.get("topo", {}).get("type"):
            return False
        if pp.get("topo", {}).get("sub") != tp.get("topo", {}).get("sub"):
            return False
    return True


def _coord_score(p_panel, t_panel):
    """
    Coord component [0,1].
    Paper: Classification of coordinate system (Cartesian, polar, 3D).
    We infer coord system from topo.sub (e.g., 'polar', 'horizontal', '3d').
    """
    p_sub = p_panel.get("topo", {}).get("sub", "")
    t_sub = t_panel.get("topo", {}).get("sub", "")
    return 1.0 if p_sub == t_sub else 0.0


def _domain_score(p_panel, t_panel):
    """
    Domain component [0,1] — Range IoU for numerical axes, exact match for categorical.
    Paper: Overlap of axis ranges or categorical domains.
    """
    p_axes = {ax["name"]: ax for ax in p_panel.get("axes", []) if isinstance(ax, dict)}
    t_axes = {ax["name"]: ax for ax in t_panel.get("axes", []) if isinstance(ax, dict)}

    if not t_axes:
        return 0.0

    axis_scores = []
    for axis_name, t_ax in t_axes.items():
        p_ax = p_axes.get(axis_name, {})
        t_dom = t_ax.get("dom", [])
        p_dom = p_ax.get("dom", [])

        if not t_dom or not p_dom:
            axis_scores.append(0.0)
            continue

        # Numerical range: compute IoU
        if isinstance(t_dom[0], (int, float)) and isinstance(p_dom[0], (int, float)):
            try:
                t_lo, t_hi = float(t_dom[0]), float(t_dom[-1])
                p_lo, p_hi = float(p_dom[0]), float(p_dom[-1])
                inter = max(0.0, min(t_hi, p_hi) - max(t_lo, p_lo))
                union = max(t_hi, p_hi) - min(t_lo, p_lo)
                axis_scores.append(inter / union if union > 0 else 0.0)
            except (ValueError, TypeError):
                axis_scores.append(0.0)
        else:
            # Categorical: Jaccard on domain lists
            t_set = set(str(v).lower() for v in t_dom)
            p_set = set(str(v).lower() for v in p_dom)
            if not t_set:
                axis_scores.append(0.0)
            else:
                jaccard = len(t_set & p_set) / len(t_set | p_set)
                axis_scores.append(jaccard)

    return sum(axis_scores) / len(axis_scores) if axis_scores else 0.0


def _series_label_score(p_panel, t_panel):
    """
    Series component [0,1] — Jaccard on legend/series names.
    Paper: Consistency of legend labels and grouping sets.
    """
    p_names = set(str(s.get("name", "")).lower() for s in p_panel.get("ser", []))
    t_names = set(str(s.get("name", "")).lower() for s in t_panel.get("ser", []))
    if not t_names:
        return 0.0
    return len(t_names & p_names) / len(t_names | p_names)


# ------------------------------------------------------------------
# COMBINED R_sem REWARD FUNCTION
# ------------------------------------------------------------------

def semantic_spec_reward_func(completions, target_specs, **kwargs):
    """
    R_sem — Semantic-Spec Reward.
    Aggregates: topology gate, coord, domain, series labels.
    Returns values in [0, 1].

    If the topology gate fails, R_sem = 0 (per paper design).
    Otherwise, the four sub-components are averaged per panel,
    then averaged across panels.
    """
    rewards = []
    for comp, target_str in zip(completions, target_specs):
        parsed = extract_json(comp)
        target = target_str if isinstance(target_str, dict) else json.loads(target_str)

        if not parsed or "panels" not in parsed or "panels" not in target:
            rewards.append(0.0)
            continue

        # Topology gate: fail = 0 for the entire R_sem
        if not _topology_passes(parsed, target):
            rewards.append(0.0)
            continue

        p_panels = parsed["panels"]
        t_panels = target["panels"]
        panel_scores = []

        for pp, tp in zip(p_panels, t_panels):
            coord  = _coord_score(pp, tp)
            domain = _domain_score(pp, tp)
            series = _series_label_score(pp, tp)
            panel_scores.append((coord + domain + series) / 3.0)

        rewards.append(sum(panel_scores) / max(1, len(panel_scores)))

    return rewards


# ------------------------------------------------------------------
# LAYER 2: CODE-SPEC REWARD  (R_code)
# Data/Func fidelity — per chart type, bounded [0,1].
# Paper: Edit Dist. / MSE for formula string matching or numeric sample error.
#
# KEY FIX: all sub-evaluators now clamp their final score to [0, 1].
# The anti-looping penalty is kept but its contribution is floored at 0
# so it can only reduce the score toward 0, never below.
# ------------------------------------------------------------------

def _safe_series_count_score(p_ser, t_ser):
    """Returns [0,1]: 1.0 for exact count match, degrades gracefully for mismatches."""
    if len(p_ser) == len(t_ser):
        return 1.0
    diff = abs(len(p_ser) - len(t_ser))
    return clamp(1.0 - 0.25 * diff)  # lose 0.25 per missing/extra series, floor 0


def _point_length_penalty(p_data, t_data):
    """Penalty in [0,1] for mismatched array lengths. 0 = no mismatch."""
    diff = abs(len(p_data) - len(t_data))
    return clamp(diff * 0.05)  # 0.05 per missing/extra point, capped at 1.0


def evaluate_bar_series(parsed_panel, target_panel):
    score = 0.0
    p_ser = parsed_panel.get("ser", [])
    t_ser = target_panel.get("ser", [])
    if not t_ser:
        return 0.0

    score += 0.3 * _safe_series_count_score(p_ser, t_ser)

    series_scores = []
    for p_s, t_s in zip(p_ser, t_ser):
        ss = 0.0
        # Series name match
        if str(p_s.get("name", "")).lower() == str(t_s.get("name", "")).lower():
            ss += 0.1

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])
        if not t_data:
            series_scores.append(clamp(ss))
            continue

        data_score = 1.0 - _point_length_penalty(p_data, t_data)
        point_scores = []
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue
            ps = 0.0
            # Categorical X
            if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                ps += 0.3
            # Primary Y
            try:
                p_y, t_y = robust_float(p_pt[1]), robust_float(t_pt[1])
                tol = max(0.05 * abs(t_y), 0.01)
                if abs(p_y - t_y) <= tol:
                    ps += 0.7
                elif abs(p_y - t_y) <= tol * 2:
                    ps += 0.35
            except ValueError:
                pass
            point_scores.append(clamp(ps))

        if point_scores:
            data_score = clamp(data_score * (sum(point_scores) / len(point_scores)))
        ss += 0.9 * data_score
        series_scores.append(clamp(ss))

    if series_scores:
        score += 0.7 * (sum(series_scores) / len(series_scores))
    return clamp(score)


def evaluate_line_series(parsed_panel, target_panel):
    p_ser = parsed_panel.get("ser", [])
    t_ser = target_panel.get("ser", [])
    if not t_ser:
        return 0.0

    score = 0.3 * _safe_series_count_score(p_ser, t_ser)
    series_scores = []

    for p_s, t_s in zip(p_ser, t_ser):
        ss = 0.0
        if str(p_s.get("name", "")).lower() == str(t_s.get("name", "")).lower():
            ss += 0.1

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])
        if not t_data:
            series_scores.append(clamp(ss))
            continue

        data_score = 1.0 - _point_length_penalty(p_data, t_data)
        point_scores = []
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue
            ps = 0.0
            # X value
            try:
                p_x, t_x = robust_float(p_pt[0]), robust_float(t_pt[0])
                if abs(p_x - t_x) <= max(0.05 * abs(t_x), 0.01):
                    ps += 0.2
            except (ValueError, TypeError):
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    ps += 0.2
            # Y value
            try:
                p_y, t_y = robust_float(p_pt[1]), robust_float(t_pt[1])
                tol = max(0.05 * abs(t_y), 0.01)
                if abs(p_y - t_y) <= tol:
                    ps += 0.6
                elif abs(p_y - t_y) <= tol * 2:
                    ps += 0.3
            except (ValueError, TypeError):
                pass
            # Extra dims (error bands, markers)
            for i in range(2, len(t_pt)):
                if i >= len(p_pt):
                    break
                try:
                    p_v, t_v = robust_float(p_pt[i]), robust_float(t_pt[i])
                    if abs(p_v - t_v) <= max(0.05 * abs(t_v), 0.01):
                        ps += 0.2
                except ValueError:
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        ps += 0.2
            point_scores.append(clamp(ps))

        if point_scores:
            data_score = clamp(data_score * (sum(point_scores) / len(point_scores)))
        ss += 0.9 * data_score
        series_scores.append(clamp(ss))

    if series_scores:
        score += 0.7 * (sum(series_scores) / len(series_scores))
    return clamp(score)


def evaluate_box_series(parsed_panel, target_panel):
    p_ser = parsed_panel.get("ser", [])
    t_ser = target_panel.get("ser", [])
    if not t_ser:
        return 0.0

    score = 0.3 * _safe_series_count_score(p_ser, t_ser)
    series_scores = []

    for p_s, t_s in zip(p_ser, t_ser):
        ss = 0.0
        if str(p_s.get("name", "")).lower() == str(t_s.get("name", "")).lower():
            ss += 0.1
        if t_s.get("is_outlier") == 1 and p_s.get("is_outlier") == 1:
            ss += 0.1

        p_data = p_s.get("data", [])
        t_data = t_s.get("data", [])
        if not t_data:
            series_scores.append(clamp(ss))
            continue

        data_score = 1.0 - _point_length_penalty(p_data, t_data)
        point_scores = []
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue
            ps = 0.0
            # Categorical X
            try:
                p_x, t_x = robust_float(p_pt[0]), robust_float(t_pt[0])
                if abs(p_x - t_x) <= max(0.05 * abs(t_x), 0.01):
                    ps += 0.2
            except (ValueError, TypeError):
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    ps += 0.2
            # Stats (5-num summary or outlier)
            stat_weight = 0.8 / max(1, len(t_pt) - 1)
            for i in range(1, len(t_pt)):
                if i >= len(p_pt):
                    break
                try:
                    p_v, t_v = robust_float(p_pt[i]), robust_float(t_pt[i])
                    tol = max(0.05 * abs(t_v), 0.01)
                    if abs(p_v - t_v) <= tol:
                        ps += stat_weight
                    elif abs(p_v - t_v) <= tol * 2:
                        ps += stat_weight * 0.4
                except ValueError:
                    if str(p_pt[i]).strip().lower() == str(t_pt[i]).strip().lower():
                        ps += stat_weight
            point_scores.append(clamp(ps))

        if point_scores:
            data_score = clamp(data_score * (sum(point_scores) / len(point_scores)))
        ss += 0.8 * data_score
        series_scores.append(clamp(ss))

    if series_scores:
        score += 0.7 * (sum(series_scores) / len(series_scores))
    return clamp(score)


def evaluate_histogram_series(parsed_panel, target_panel):
    p_ser = parsed_panel.get("ser", [])
    t_ser = target_panel.get("ser", [])
    if not t_ser:
        return 0.0

    score = 0.3 * _safe_series_count_score(p_ser, t_ser)
    series_scores = []

    for p_s, t_s in zip(p_ser, t_ser):
        ss = 0.0
        if str(p_s.get("name", "")).lower() == str(t_s.get("name", "")).lower():
            ss += 0.1

        is_kde = t_s.get("is_kde") == 1 or t_s.get("type") == "density_curve"
        if is_kde:
            if p_s.get("is_kde") == 1:
                ss += 0.1
            p_data, t_data = p_s.get("points", []), t_s.get("points", [])
        else:
            if t_s.get("is_aggregated") == 1 and p_s.get("is_aggregated") == 1:
                ss += 0.1
            p_data, t_data = p_s.get("data", []), t_s.get("data", [])

        if not t_data:
            series_scores.append(clamp(ss))
            continue

        data_score = 1.0 - _point_length_penalty(p_data, t_data)
        point_scores = []
        for p_pt, t_pt in zip(p_data, t_data):
            if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                continue
            ps = 0.0
            try:
                p_x, t_x = robust_float(p_pt[0]), robust_float(t_pt[0])
                if abs(p_x - t_x) <= max(0.05 * abs(t_x), 0.01):
                    ps += 0.3
            except ValueError:
                if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                    ps += 0.3
            try:
                p_y, t_y = robust_float(p_pt[1]), robust_float(t_pt[1])
                tol = max(0.05 * abs(t_y), 0.01)
                if abs(p_y - t_y) <= tol:
                    ps += 0.7
                elif abs(p_y - t_y) <= tol * 2:
                    ps += 0.35
            except ValueError:
                pass
            point_scores.append(clamp(ps))

        if point_scores:
            data_score = clamp(data_score * (sum(point_scores) / len(point_scores)))
        ss += 0.8 * data_score
        series_scores.append(clamp(ss))

    if series_scores:
        score += 0.7 * (sum(series_scores) / len(series_scores))
    return clamp(score)


def evaluate_scatter_series(parsed_panel, target_panel):
    p_ser = parsed_panel.get("ser", [])
    t_ser = target_panel.get("ser", [])
    if not t_ser:
        return 0.0

    score = 0.3 * _safe_series_count_score(p_ser, t_ser)
    series_scores = []

    for p_s, t_s in zip(p_ser, t_ser):
        ss = 0.0
        if str(p_s.get("name", "")).lower() == str(t_s.get("name", "")).lower():
            ss += 0.1
        if str(p_s.get("y_ref")) == str(t_s.get("y_ref")):
            ss += 0.1

        if t_s.get("is_clustered") == 1:
            if p_s.get("is_clustered") == 1:
                ss += 0.1
            t_clusters = t_s.get("cluster_descriptions", [])
            p_clusters = p_s.get("cluster_descriptions", [])
            cluster_scores = []
            for p_c, t_c in zip(p_clusters, t_clusters):
                if not isinstance(p_c, dict) or not isinstance(t_c, dict):
                    continue
                cs = 0.0
                try:
                    for dim_i, dim_tol in enumerate([0.05, 0.05]):
                        p_v = robust_float(p_c.get("centroid", [0, 0])[dim_i])
                        t_v = robust_float(t_c.get("centroid", [0, 0])[dim_i])
                        if abs(p_v - t_v) <= max(dim_tol * abs(t_v), 0.01):
                            cs += 0.25
                except (ValueError, TypeError, IndexError):
                    pass
                cluster_scores.append(clamp(cs))
            if cluster_scores:
                ss += 0.7 * (sum(cluster_scores) / len(cluster_scores))
        else:
            p_data, t_data = p_s.get("data", []), t_s.get("data", [])
            if not t_data:
                series_scores.append(clamp(ss))
                continue
            data_score = 1.0 - _point_length_penalty(p_data, t_data)
            point_scores = []
            for p_pt, t_pt in zip(p_data, t_data):
                if not isinstance(p_pt, list) or len(p_pt) < 2 or len(t_pt) < 2:
                    continue
                ps = 0.0
                try:
                    p_x, t_x = robust_float(p_pt[0]), robust_float(t_pt[0])
                    if abs(p_x - t_x) <= max(0.05 * abs(t_x), 0.01):
                        ps += 0.3
                except ValueError:
                    if str(p_pt[0]).strip().lower() == str(t_pt[0]).strip().lower():
                        ps += 0.3
                try:
                    p_y, t_y = robust_float(p_pt[1]), robust_float(t_pt[1])
                    tol = max(0.05 * abs(t_y), 0.01)
                    ps += 0.7 if abs(p_y - t_y) <= tol else (0.35 if abs(p_y - t_y) <= tol * 2 else 0.0)
                except ValueError:
                    pass
                point_scores.append(clamp(ps))
            if point_scores:
                data_score = clamp(data_score * (sum(point_scores) / len(point_scores)))
            ss += 0.8 * data_score

        series_scores.append(clamp(ss))

    if series_scores:
        score += 0.7 * (sum(series_scores) / len(series_scores))
    return clamp(score)


def evaluate_pie_series(parsed_panel, target_panel):
    p_ser = parsed_panel.get("ser", [])
    t_ser = target_panel.get("ser", [])
    if not t_ser:
        return 0.0

    count_score = _safe_series_count_score(p_ser, t_ser)
    slice_scores = []

    for p_s, t_s in zip(p_ser, t_ser):
        ss = 0.0
        if str(p_s.get("name", "")).lower() == str(t_s.get("name", "")).lower():
            ss += 0.2
        p_data, t_data = p_s.get("data", []), t_s.get("data", [])
        if p_data and t_data:
            p_pt = p_data[0] if isinstance(p_data[0], list) else []
            t_pt = t_data[0] if isinstance(t_data[0], list) else []
            if p_pt and t_pt:
                try:
                    p_pct, t_pct = robust_float(p_pt[0]), robust_float(t_pt[0])
                    tol = max(0.02 * abs(t_pct), 0.005)
                    ss += 0.5 if abs(p_pct - t_pct) <= tol else (0.2 if abs(p_pct - t_pct) <= tol * 2 else 0.0)
                except (ValueError, TypeError):
                    pass
                if len(t_pt) > 1 and len(p_pt) > 1:
                    try:
                        p_v, t_v = robust_float(p_pt[1]), robust_float(t_pt[1])
                        tol = max(0.05 * abs(t_v), 0.01)
                        ss += 0.3 if abs(p_v - t_v) <= tol else 0.0
                    except (ValueError, TypeError):
                        pass
        slice_scores.append(clamp(ss))

    if not slice_scores:
        return 0.0
    return clamp(0.3 * count_score + 0.7 * (sum(slice_scores) / len(slice_scores)))


# ------------------------------------------------------------------
# COMBINED R_code REWARD FUNCTION
# ------------------------------------------------------------------

CHART_EVALUATORS = {
    "bar":       evaluate_bar_series,
    "line":      evaluate_line_series,
    "box":       evaluate_box_series,
    "histogram": evaluate_histogram_series,
    "scatter":   evaluate_scatter_series,
    "pie":       evaluate_pie_series,
}


def data_fidelity_reward_func(completions, target_specs, **kwargs):
    """
    R_code — Code-Spec / Data Fidelity Reward.
    Routes to the per-chart-type evaluator and normalises by panel count.
    Returns values in [0, 1].

    Topology gate is re-checked here: if topology fails, R_code = 0.
    This mirrors the paper's hierarchical gating.
    """
    rewards = []
    for comp, target_str in zip(completions, target_specs):
        parsed = extract_json(comp)
        target = target_str if isinstance(target_str, dict) else json.loads(target_str)

        if not parsed or "panels" not in parsed or "panels" not in target:
            rewards.append(0.0)
            continue

        # Gate: wrong topology → no data fidelity reward
        if not _topology_passes(parsed, target):
            rewards.append(0.0)
            continue

        panel_scores = []
        for pp, tp in zip(parsed["panels"], target["panels"]):
            chart_type = tp.get("topo", {}).get("type", "")
            evaluator = CHART_EVALUATORS.get(chart_type)
            panel_scores.append(evaluator(pp, tp) if evaluator else 0.0)

        rewards.append(sum(panel_scores) / max(1, len(panel_scores)))

    return rewards

In [ ]:
# ==========================================
# 3. REWARD SANITY TESTS
# ==========================================
import json

target_spec = {
    "title": "Usage Frequency of Transportation Modes",
    "panel_count": 2, "panel_layout": [2, 1],
    "legend": {"on": 0},
    "panels": [
        {
            "panel_idx": 0, "panel_type": "bar", "title": "Usage Frequency", "has_errorbar": 0,
            "topo": {"type": "bar", "sub": "horizontal"},
            "axes": [
                {"name": "x_axis", "scl": "linear", "dom": [0.0, 45]},
                {"name": "y_axis", "scl": "categorical", "dom": ["Walk", "Train", "Bus", "Car", "Bike"]}
            ],
            "ser": [{"name": "bar_series_1", "color": "#ff7f0e", "symbol": "none",
                     "data": [["Walk", 5], ["Train", 10], ["Bus", 20], ["Car", 30], ["Bike", 40]]}]
        },
        {
            "panel_idx": 1, "panel_type": "bar", "has_errorbar": 0,
            "topo": {"type": "bar", "sub": "horizontal"},
            "axes": [
                {"name": "x_axis", "lbl": "Frequency of Use", "scl": "linear", "dom": [0.0, 45]},
                {"name": "y_axis", "scl": "categorical",
                 "dom": ["Walk (New)", "Train (New)", "Bus (New)", "Car (New)", "Bike (New)"]}
            ],
            "ser": [{"name": "Frequency of Use", "color": "#ff7f0e", "symbol": "none",
                     "data": [["Walk (New)", 7], ["Train (New)", 12], ["Bus (New)", 18],
                              ["Car (New)", 25], ["Bike (New)", 35]]}]
        }
    ],
    "chart_type": "bar"
}

target_str = json.dumps(target_spec)

# Perfect match
completion_perfect = f"```json\n{target_str}\n```"

# Flawed: truncated data, wrong series name
bad_spec = json.loads(target_str)
bad_spec["panels"][0]["ser"][0]["data"].pop()
bad_spec["panels"][1]["ser"][0]["name"] = "Wrong Name"
completion_flawed = f"```json\n{json.dumps(bad_spec)}\n```"

# Wrong topology (chart_type mismatch)
topo_fail = json.loads(target_str)
topo_fail["chart_type"] = "line"
topo_fail["panels"][0]["topo"]["type"] = "line"
topo_fail["panels"][1]["topo"]["type"] = "line"
completion_topo_fail = f"```json\n{json.dumps(topo_fail)}\n```"

# Broken JSON
completion_invalid = "```json\n{ bad json...\n```"

completions   = [completion_perfect, completion_flawed, completion_topo_fail, completion_invalid]
target_specs  = [target_str] * 4

print("R_f  Format  (expect [0.0, 0.0, 0.0, -2.0]):")
print(format_reward_func(completions))

print("\nR_e  Schema  (expect [0.5, 0.5, 0.5, 0.0]):")
print(schema_reward_func(completions))

print("\nR_sem Semantic (expect [~1.0, partial, 0.0, 0.0] — topo gate kills 3rd):")
print(semantic_spec_reward_func(completions, target_specs=target_specs))

print("\nR_code Data Fidelity (expect [~1.0, partial, 0.0, 0.0]):")
print(data_fidelity_reward_func(completions, target_specs=target_specs))

In [ ]:
import json
import os
from datasets import Dataset

WORKSPACE_DIR = "/workspace/ImageToSpec_Stage1"
LOCAL_IMAGE_DIR = os.path.join(WORKSPACE_DIR, "final_balanced_images")


def load_grpo_dataset(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed_data = []
    for item in data:
        raw_img_path = item["images"][0].replace("\\", "/")
        img_filename = os.path.basename(raw_img_path)
        local_img_path = os.path.join(LOCAL_IMAGE_DIR, img_filename)

        target_spec_str = item["messages"][1]["content"]

        processed_data.append({
            "id": str(item["id"]),
            "messages": item["messages"],
            "images": [local_img_path],
            "target_specs": target_spec_str
        })

    return Dataset.from_list(processed_data)


hf_dataset = load_grpo_dataset(os.path.join(WORKSPACE_DIR, "qwen25_vl_3b_train.json"))
print(f"Loaded {len(hf_dataset)} GRPO samples.")
print(f"Dataset Columns: {hf_dataset.column_names}")

In [ ]:
# ==========================================
# FORMAT DATASET FOR GRPO
# ==========================================
def prepare_dataset_for_grpo(example):
    user_message = [msg for msg in example["messages"] if msg["role"] == "user"]
    assistant_message = [msg for msg in example["messages"] if msg["role"] == "assistant"]
    target_spec = assistant_message[0]["content"] if assistant_message else ""

    clean_user_message = []
    for msg in user_message:
        content = msg.get("content", "")
        if isinstance(content, str) and "<|image_pad|>" in content:
            text_part = content.replace("<|image_pad|>", "").strip()
            clean_content = [
                {"type": "image"},
                {"type": "text", "text": text_part},
            ]
            clean_user_message.append({"role": "user", "content": clean_content})
        else:
            clean_user_message.append(msg)

    return {
        "prompt": clean_user_message,
        "images": example["images"],
        "target_specs": target_spec,
    }

print("Formatting dataset for GRPO...")
hf_dataset = hf_dataset.map(
    prepare_dataset_for_grpo,
    remove_columns=hf_dataset.column_names
)
print("Columns available:", hf_dataset.column_names)

In [ ]:
from PIL import Image
import os

print("Validating image paths...")
bad = []
for i, ex in enumerate(hf_dataset):
    path = ex["images"][0]
    if not os.path.exists(path):
        bad.append((i, path))
    else:
        try:
            Image.open(path).verify()
        except Exception as e:
            bad.append((i, f"{path} — {e}"))

if bad:
    print(f"❌ Found {len(bad)} bad images:")
    for b in bad: print(" ", b)
    bad_indices = {b[0] for b in bad}
    hf_dataset = hf_dataset.filter(lambda _, i: i not in bad_indices, with_indices=True)
    print(f"✅ Dataset cleaned: {len(hf_dataset)} samples remaining")
else:
    print("✅ All images valid")

In [ ]:
import json
with open("/workspace/ImageToSpec_Stage1/qwen25_vl_3b_train.json") as f:
    raw = json.load(f)

print(json.dumps(raw[0]["messages"][0], indent=2))

In [ ]:
import torch
from unsloth import FastVisionModel, is_bf16_supported
from trl import GRPOConfig, GRPOTrainer

# 1. Load Model & Processor
model, processor = FastVisionModel.from_pretrained(
    "/workspace/ImageToSpec_Stage1/qwen3b_lora_sft_bf16_final/qwen3b_lora_sft_bf16_final",
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

# ==========================================
# GRPO CONFIGURATION
#
# KEY CHANGES AND RATIONALE:
#
# beta: 0.001 → 0.04
#   The old value was far too small, giving KL almost zero weight in the
#   objective.  With all rewards now bounded [0,1], the policy can still
#   earn meaningful signal at beta=0.04 while staying close to the SFT
#   reference.  Start here; if KL still drifts above ~5, raise to 0.1.
#
# max_completion_length: 3072 → 1536
#   Chart specs are JSON objects, not long chains of reasoning.  3072
#   encouraged length and caused frequent max-length truncations which
#   broke the JSON parser → format_reward = -2 → large gradient spikes.
#   1536 comfortably covers even complex multi-panel specs.
#   Raise back toward 2048 only if you observe real truncations.
#
# max_prompt_length: added at 1024
#   Without this, very long image+instruction prompts get silently
#   truncated in unpredictable ways.  1024 is a safe ceiling for the
#   text portion of a chart description prompt.
#
# learning_rate: unchanged at 4e-7 (matches paper)
# num_generations: unchanged at 8
# ==========================================
training_args = GRPOConfig(
    output_dir="/workspace/unsloth_qwen3b_grpo",

    num_generations=8,
    num_train_epochs=1,

    # FIX 1: Halved completion length to stop truncation → parse failures
    max_completion_length=3072,
    # FIX 2: Explicit prompt length cap to prevent silent over-truncation
    max_prompt_length=1024,

    # FIX 3: Raised beta so KL term actually constrains the policy update
    # With fully bounded [0,1] rewards, 0.04 gives stable training.
    # If you see KL > 10 after a few hundred steps, raise to 0.1.
    beta=0.04,

    learning_rate=4e-7,          # Paper: 4×10⁻⁷
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=is_bf16_supported(),
    optim="adamw_8bit",

    logging_steps=5,
    save_steps=50,
)

# 3. Initialize GRPOTrainer with the paper-aligned 4-function reward stack
#
# Function mapping to paper Table 2:
#   format_reward_func        → R_f  (Integrity: Format)
#   schema_reward_func        → R_e  (Integrity: Execution/Schema)
#   semantic_spec_reward_func → R_sem (Topology gate + Coord + Domain + Series)
#   data_fidelity_reward_func → R_code (Statistical/Relational/Vector/Auxiliary)
trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[
        format_reward_func,
        schema_reward_func,
        semantic_spec_reward_func,
        data_fidelity_reward_func,
    ],
    train_dataset=hf_dataset,
    args=training_args,
)

print("Starting GRPO Training...")
trainer.train()

model.save_pretrained("/workspace/unsloth_qwen3b_grpo_final")
processor.save_pretrained("/workspace/unsloth_qwen3b_grpo_final")